# RAG with HuggingFace and Milvus -  Notebook

we will implement a complete RAG (Retrieval-Augmented Generation) pipeline using:
- **Dataset**: HuggingFace Documentation (`m-ric/huggingface_doc`)
- **Vector Store**: Milvus
- **Embeddings**: BGE-small-en-v1.5
- **LLM**: Microsoft Phi-3-mini-4k-instruct/"Qwen/Qwen2-1.5B-Instruct"
- **Evaluation**: Opik (AnswerRelevance, Hallucination)

## Instructions
1. Read through each section carefully
2. Complete the code in cells marked with `# TODO`
3. Run all cells in order
4. Verify your implementation with the evaluation cells


---

## 1. Setup

Install required dependencies and configure environment.

In [ ]:
# Install ONE consistent dependency set.
# IMPORTANT: after this cell finishes, restart the Colab runtime/kernel once.
!pip install -q --no-cache-dir \
    "transformers==4.48.3" \
    "tokenizers==0.21.0" \
    "accelerate==1.4.0" \
    "sentence-transformers==3.4.1" \
    "pymilvus[milvus_lite]" \
    "datasets" \
    "opik" \
    "tqdm"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 264.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.6/158.6 kB 366.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 183.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 268.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 322.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 393.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 359.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 175.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 274.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 327.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 330.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Verify the environment after restarting the runtime/kernel.
import sys
import torch
import transformers
import tokenizers
import accelerate
import sentence_transformers
import pymilvus

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Accelerate:", accelerate.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("PyMilvus:", pymilvus.__version__)
print("CUDA:", torch.cuda.is_available())

assert transformers.__version__ == "4.48.3", f"Expected Transformers 4.48.3, got {transformers.__version__}"
assert tokenizers.__version__ == "0.21.0", f"Expected Tokenizers 0.21.0, got {tokenizers.__version__}"
print("✅ Dependency versions are consistent.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu128
Transformers: 4.48.3
Tokenizers: 0.21.0
Accelerate: 1.4.0
Sentence Transformers: 3.4.1
PyMilvus: 3.0.1
CUDA: True
✅ Dependency versions are consistent.


In [ ]:
import os
from getpass import getpass

# ============================================================
# SECURE API KEY INPUT
# ============================================================
# We avoid google.colab.userdata here because its secret bridge can
# time out even when a secret exists. Enter keys interactively instead.

HF_TOKEN = getpass("Enter Hugging Face token (press Enter to skip): ").strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF_TOKEN configured.")
else:
    os.environ.pop("HF_TOKEN", None)
    print("ℹ️ HF_TOKEN skipped; public Hugging Face resources can still be used.")

OPIK_API_KEY = getpass("Enter Opik API key (press Enter to skip): ").strip()
if OPIK_API_KEY:
    os.environ["OPIK_API_KEY"] = OPIK_API_KEY
    print("✅ OPIK_API_KEY configured.")
else:
    os.environ.pop("OPIK_API_KEY", None)
    print("ℹ️ OPIK_API_KEY skipped; Opik evaluation will be skipped.")



Enter Hugging Face token (press Enter to skip): ··········
✅ HF_TOKEN configured.
Enter Opik API key (press Enter to skip): ··········
✅ OPIK_API_KEY configured.


In [ ]:
# Optional LLM-as-a-judge credential. Opik's AnswerRelevance/Hallucination
# metrics default to an external judge model (commonly GPT-4o).
OPENAI_API_KEY = getpass("Enter OpenAI API key for Opik judge (press Enter to skip): ").strip()
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("✅ OPENAI_API_KEY configured for Opik judge.")
else:
    os.environ.pop("OPENAI_API_KEY", None)
    print("ℹ️ OPENAI_API_KEY skipped; Opik LLM-as-judge cells will be skipped.")

Enter OpenAI API key for Opik judge (press Enter to skip): ··········
✅ OPENAI_API_KEY configured for Opik judge.


In [ ]:
print("HF_TOKEN configured:", bool(os.environ.get("HF_TOKEN")))
print("OPIK_API_KEY configured:", bool(os.environ.get("OPIK_API_KEY")))
print("OPENAI_API_KEY configured:", bool(os.environ.get("OPENAI_API_KEY")))


HF_TOKEN configured: True
OPIK_API_KEY configured: True
OPENAI_API_KEY configured: True


## 2. Data Loading

Load the HuggingFace documentation dataset.

In [ ]:
from datasets import load_dataset

# Load the HuggingFace documentation dataset
dataset = load_dataset("m-ric/huggingface_doc", split="train")

print(f"Dataset loaded with {len(dataset)} documents")
print(f"Columns: {dataset.column_names}")
print(f"\nSample document (first 500 chars):")
print(dataset[0]["text"][:500])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Dataset loaded with 2647 documents
Columns: ['text', 'source']

Sample document (first 500 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-docu


In [ ]:
# Extract text and source information
documents = []
for item in dataset:
    documents.append({"text": item["text"],"source": item["source"]})

print(f"Extracted {len(documents)} documents")

# For this assignment, we'll use a subset to keep things manageable
MAX_DOCS = 500
documents = documents[:MAX_DOCS]
print(f"Using {len(documents)} documents for this assignment")

Extracted 2647 documents
Using 500 documents for this assignment


## 3. Chunking

Split documents into smaller chunks for better retrieval.

### Your Task
Implement the `chunk_document` function that:
1. Takes a text string, chunk_size, and chunk_overlap as parameters
2. Splits the text into overlapping chunks of the specified size
3. Returns a list of chunk strings

### Hints
- Use a sliding window approach with step = chunk_size - chunk_overlap
- Handle edge cases: empty text, text shorter than chunk_size
- Make sure each chunk is non-empty before adding it

In [ ]:
doc_lengths = [len(d) for d in dataset["text"]]
print(f"\nDocument length (chars) — min: {min(doc_lengths)}, "
      f"max: {max(doc_lengths)}, mean: {sum(doc_lengths)/len(doc_lengths):.0f}, "
      f"median: {sorted(doc_lengths)[len(doc_lengths)//2]}")


Document length (chars) — min: 23, max: 371057, mean: 8079, median: 4575


In [ ]:
from typing import List, Dict

def chunk_document(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """Split text into overlapping string chunks using a sliding window."""
    if text is None or text == "":
        return []
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be >= 0 and < chunk_size")
    if len(text) <= chunk_size:
        return [text] if text.strip() else []
    step = chunk_size - chunk_overlap
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk)
        if end == len(text):
            break
        start += step
    return chunks


def chunk_all_documents(documents: List[Dict], chunk_size: int = 1000, chunk_overlap: int = 200) -> List[Dict]:
    """Chunk all documents and preserve source, document, chunk, and character-offset lineage."""
    all_chunks = []
    chunk_id = 0
    for doc_index, document in enumerate(documents):
        text = document.get("text", "")
        source = document.get("source", "")
        doc_id = f"doc-{doc_index}"
        doc_chunks = chunk_document(text, chunk_size, chunk_overlap)
        for chunk_index, chunk in enumerate(doc_chunks):
            char_start = min(chunk_index * (chunk_size - chunk_overlap), len(text))
            char_end = min(char_start + len(chunk), len(text))
            all_chunks.append({
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "chunk_index": chunk_index,
                "text": chunk,
                "source": source,
                "char_start": char_start,
                "char_end": char_end,
            })
            chunk_id += 1
    return all_chunks


In [ ]:
def dedupe_boilerplate(chunks, min_unique_len=40, min_doc_spread=5):
    """
    Drop chunks whose text prefix appears near-identically across many
    *different* source documents (a signature of license headers, footers,
    or repeated templated sections) rather than being genuinely unique content.
    """
    from collections import defaultdict

    prefix_to_docs = defaultdict(set)
    for c in chunks:
        prefix = c["text"].strip()[:min_unique_len]
        prefix_to_docs[prefix].add(c["doc_id"])

    boilerplate_prefixes = {
        prefix for prefix, doc_ids in prefix_to_docs.items()
        if len(doc_ids) > min_doc_spread
    }

    filtered = [
        c for c in chunks
        if c["text"].strip()[:min_unique_len] not in boilerplate_prefixes
    ]

    removed = len(chunks) - len(filtered)
    print(f"Removed {removed} boilerplate-like chunks "
          f"({len(boilerplate_prefixes)} distinct boilerplate prefixes, "
          f"spread across >{min_doc_spread} docs each)")
    return filtered

In [ ]:
# Test chunk size and overlap
TEST_CHUNK_SIZE = 1000
TEST_OVERLAP = 200
test_text = "A" * 2500
test_chunks = chunk_document(test_text, TEST_CHUNK_SIZE, TEST_OVERLAP)
print(f"Test chunks: {len(test_chunks)}")
assert 3 <= len(test_chunks) <= 4
assert all(len(chunk) <= TEST_CHUNK_SIZE for chunk in test_chunks)
assert test_chunks[0][-TEST_OVERLAP:] == test_chunks[1][:TEST_OVERLAP]
print("✅ Chunk size and overlap tests passed!")
lineage_test = chunk_all_documents([{"text": test_text, "source": "test.md"}], TEST_CHUNK_SIZE, TEST_OVERLAP)
assert lineage_test[0]["source"] == "test.md"
assert lineage_test[0]["doc_id"] == "doc-0"
assert lineage_test[1]["chunk_index"] == 1
assert lineage_test[1]["char_start"] == TEST_CHUNK_SIZE - TEST_OVERLAP
print("✅ Metadata lineage test passed!")


Test chunks: 3
✅ Chunk size and overlap tests passed!
✅ Metadata lineage test passed!


In [ ]:
# Create chunks from all documents
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = chunk_all_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nCreated {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunks per document: {len(chunks) / len(documents):.2f}")

# NEW CELL — insert here, before embeddings
chunks = dedupe_boilerplate(chunks)
print(f"Chunks after dedup: {len(chunks)}")

print(f"Average chunks per document after dedup: {len(chunks) / len(documents):.2f}")

# Show sample chunk
if chunks:
    print(f"\nSample chunk:")
    print(f"  ID: {chunks[0]['chunk_id']}")
    print(f"  Source: {chunks[0]['source']}")
    print(f"  Text (first 200 chars): {chunks[0]['text'][:200]}...")


# Quick sanity check — inspect what got flagged as boilerplate
sample_removed = [c for c in chunk_all_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)
                   if c["chunk_id"] not in {x["chunk_id"] for x in chunks}][:3]
for c in sample_removed:
    print(f"[{c['source']}] {c['text'][:80]!r}")


Created 5535 chunks from 500 documents
Average chunks per document: 11.07
Removed 157 boilerplate-like chunks (8 distinct boilerplate prefixes, spread across >5 docs each)
Chunks after dedup: 5378
Average chunks per document after dedup: 10.76

Sample chunk:
  ID: 0
  Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
  Text (first 200 chars):  Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deplo...
[huggingface/transformers/blob/main/docs/source/en/perf_train_tpu_tf.md] '!--Copyright 2023 The HuggingFace Team. All rights reserved.\n\nLicensed under the'
[huggingface/transformers/blob/main/docs/source/en/model_doc/vision-text-dual-encoder.md] '!--Copyright 2021 The HuggingFace Team. All rights reserved.\n\nLicensed under the'
[huggingface/diffusers/blob/main/docs/source/en/api/pipelines/kandinsky

## 4. Embeddings

Generate vector embeddings for each chunk using BGE-small-en-v1.5.

### Your Task
Implement the `generate_embeddings` function that:
1. Processes texts in batches for memory efficiency
2. Uses the SentenceTransformer model to generate embeddings
3. Returns embeddings as a list of lists (for Milvus compatibility)

### Hints
- Use `model.encode()` with `normalize_embeddings=True` for cosine similarity
- Process in batches to avoid memory issues
- Convert numpy arrays to lists using `.tolist()`

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" #Use any model of your choice from Sentence Transformers
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Loaded embedding model: {EMBEDDING_MODEL}")

# Test embedding
test_embedding = embedding_model.encode(["This is a test"], normalize_embeddings=True)
EMBEDDING_DIM = len(test_embedding[0])
print(f"Embedding dimension: {EMBEDDING_DIM}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: BAAI/bge-small-en-v1.5
Embedding dimension: 384


In [ ]:
from tqdm import tqdm
# ============================================================
# TODO: IMPLEMENT EMBEDDING GENERATION
# ============================================================

def generate_embeddings(texts: List[str], model: SentenceTransformer, batch_size: int = 32) -> List[List[float]]:
    """
    Generate embeddings for a list of texts.
    Args:
        texts: List of text strings to embed
        model: SentenceTransformer model
        batch_size: Number of texts to process at once
    Returns:
        List of embedding vectors (as lists of floats)
    """
    all_embeddings = []

    # TODO: Generate embeddings in batches
    # Generate embeddings in batches
    #for start in range(0, len(texts), batch_size):
    for start in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        # Get the current batch
        batch = texts[start:start + batch_size]

        # Generate normalized embeddings
        embeddings = model.encode(batch,normalize_embeddings=True,show_progress_bar=False)

        # Convert numpy array to Python lists
        all_embeddings.extend(embeddings.tolist())

    return all_embeddings

In [ ]:
# Test your embedding generation
test_texts = ["Hello world", "This is a test", "RAG is cool"]
test_embeddings = generate_embeddings(test_texts, embedding_model)

print(f"Generated {len(test_embeddings)} embeddings")
print(f"Embedding dimension: {len(test_embeddings[0]) if test_embeddings else 0}")

if len(test_embeddings) == 3 and len(test_embeddings[0]) == 384:
    print("✅ Embedding generation test passed!")
else:
    print("❌ Check your embedding implementation")

Embedding: 100%|██████████| 1/1 [00:00<00:00,  8.03it/s]

Generated 3 embeddings
Embedding dimension: 384
✅ Embedding generation test passed!


In [ ]:
# Generate embeddings for all chunks
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = generate_embeddings(chunk_texts, embedding_model)

print(f"\nGenerated {len(embeddings)} embeddings")
if embeddings:
    print(f"Embedding dimension: {len(embeddings[0])}")
    print(f"Sample embedding (first 10 values): {embeddings[0][:10]}")

Embedding: 100%|██████████| 169/169 [00:45<00:00,  3.73it/s]


Generated 5378 embeddings
Embedding dimension: 384
Sample embedding (first 10 values): [-0.07532959431409836, -0.027507992461323738, -0.03995613381266594, -0.040492136031389236, 0.033340033143758774, 0.04296518489718437, -0.043336279690265656, -0.04493821784853935, -0.05554318055510521, 0.02672027423977852]


## 5. Vector Store (Milvus)

Store embeddings in Milvus for efficient similarity search.

### Your Task
1. Implement `setup_milvus_collection` to create a new collection
2. Implement `insert_data_to_milvus` to insert chunks and embeddings

### Hints
- Use `client.has_collection()` to check if collection exists
- Use `client.drop_collection()` to remove existing collection
- Use `client.create_collection()` with dimension and metric_type parameters
- Use `client.insert()` to add data

In [ ]:
from pymilvus import MilvusClient

# Initialize Milvus client (uses Milvus Lite - stores data locally)
MILVUS_DB_PATH = "./hf_docs_milvus.db"
milvus_client = MilvusClient(uri=MILVUS_DB_PATH)
COLLECTION_NAME = "hf_documentation"
print(f"Milvus client initialized with database: {MILVUS_DB_PATH}")

Milvus client initialized with database: ./hf_docs_milvus.db


In [ ]:
# ============================================================
# MILVUS COLLECTION SETUP
# ============================================================
# Embeddings are L2-normalized. With normalized vectors, Inner Product
# is equivalent to cosine similarity, matching the assignment rubric.

from pymilvus import DataType

def setup_milvus_collection(client: MilvusClient, collection_name: str, embedding_dim: int):
    """Create a persistent Milvus collection with explicit schema and IP metric."""
    if client.has_collection(collection_name):
        client.drop_collection(collection_name)
        print(f"Dropped existing collection: {collection_name}")

    schema = client.create_schema(auto_id=False, enable_dynamic_field=False)

    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=embedding_dim)
    schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=65535)
    schema.add_field(field_name="source", datatype=DataType.VARCHAR, max_length=2048)
    schema.add_field(field_name="doc_id", datatype=DataType.VARCHAR, max_length=128)
    schema.add_field(field_name="chunk_index", datatype=DataType.INT64)
    schema.add_field(field_name="char_start", datatype=DataType.INT64)
    schema.add_field(field_name="char_end", datatype=DataType.INT64)

    # Vector index
    index_params = client.prepare_index_params()

    index_params.add_index(field_name="vector",index_type="AUTOINDEX",metric_type="IP",)

    client.create_collection(collection_name=collection_name,schema=schema,
        index_params=index_params,
    )

    print(f"Created collection: {collection_name} with IP metric and dimension {embedding_dim}")
    client.load_collection(collection_name)

    print()
    print(f"✅ Collection created: {collection_name}")
    print(f"✅ Vector dimension: {embedding_dim}")
    print("✅ Vector metric: IP")
    print("✅ Vector index: AUTOINDEX")
    print("✅ Scalar STL_SORT index: NOT USED")
    print("✅ Dynamic fields: disabled")
    print("✅ Metadata fields: doc_id, chunk_index, char_start, char_end")


In [ ]:
# Setup the collection
setup_milvus_collection(milvus_client, COLLECTION_NAME, EMBEDDING_DIM)

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
  File "/usr/local/lib/python3.13/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1264, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Created collection: hf_documentation with IP metric and dimension 384

✅ Collection created: hf_documentation
✅ Vector dimension: 384
✅ Vector metric: IP
✅ Vector index: AUTOINDEX
✅ Scalar STL_SORT index: NOT USED
✅ Dynamic fields: disabled
✅ Metadata fields: doc_id, chunk_index, char_start, char_end


In [ ]:
# ===============================
# TODO: IMPLEMENT DATA INSERTION
# ===============================

def insert_data_to_milvus(
    client: MilvusClient,
    collection_name: str,
    chunks: List[Dict],
    embeddings: List[List[float]],
    batch_size: int = 100
):
    """Insert chunk records in batches while preserving lineage metadata."""
    if len(chunks) != len(embeddings):
        raise ValueError("chunks and embeddings must have the same length")

    total_inserted = 0
    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        batch = []
        for chunk, embedding in zip(chunks[start:end], embeddings[start:end]):
            batch.append({
                "id": chunk["chunk_id"],
                "vector": embedding,
                "text": chunk["text"],
                "source": chunk["source"],
                "doc_id": chunk["doc_id"],
                "chunk_index": chunk["chunk_index"],
                "char_start": chunk["char_start"],
                "char_end": chunk["char_end"],
            })

        result = client.insert(collection_name=collection_name, data=batch)
        total_inserted += result["insert_count"]

    return total_inserted


In [ ]:
inserted_count = insert_data_to_milvus(milvus_client, COLLECTION_NAME, chunks, embeddings)
print(f"Inserted {inserted_count} records into Milvus")
assert inserted_count == len(chunks)

# Persistence checks
collection_stats = milvus_client.get_collection_stats(collection_name=COLLECTION_NAME)
print("Collection stats:", collection_stats)
print("Milvus DB file exists:", os.path.exists(MILVUS_DB_PATH))
assert os.path.exists(MILVUS_DB_PATH)
print("✅ All chunks inserted and local Milvus persistence verified!")


Inserted 5378 records into Milvus
Collection stats: {'row_count': 5378}
Milvus DB file exists: True
✅ All chunks inserted and local Milvus persistence verified!


## 6. Retrieval

Implement semantic search to retrieve relevant documents for a query.

### Your Task
Implement the `retrieve_documents` function that:
1. Generates an embedding for the query
2. Searches Milvus for similar vectors
3. Returns the top-K most relevant documents

### Hints
- Use `embedding_model.encode()` to embed the query
- Use `client.search()` to find similar vectors
- Extract text and source from the search results

In [ ]:
# ============================================================
# TODO: IMPLEMENT RETRIEVAL (25 points)
# ============================================================

def retrieve_documents(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    top_k: int = 5
) -> List[Dict]:
    """Retrieve semantically similar chunks using normalized-vector IP search."""
    if not query.strip():
        return []
    if top_k <= 0:
        raise ValueError("top_k must be > 0")

    query_embedding = embedding_model.encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).tolist()[0]

    search_results = client.search(
        collection_name=collection_name,
        data=[query_embedding],
        limit=top_k,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=[
            "text", "source", "doc_id", "chunk_index", "char_start", "char_end"
        ],
    )

    retrieved_docs = []
    for result in search_results[0]:
        entity = result["entity"]
        retrieved_docs.append({
            "chunk_id": result["id"],
            "text": entity["text"],
            "source": entity["source"],
            "doc_id": entity["doc_id"],
            "chunk_index": entity["chunk_index"],
            "char_start": entity["char_start"],
            "char_end": entity["char_end"],
            "score": result["distance"],
        })
    return retrieved_docs


In [ ]:
# Test retrieval
test_query = "How do I fine-tune a transformer model?"

retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved):
    print(f"\n--- Document {i+1} (Score: {doc.get('score', 'N/A')}) ---")
    print(f"Source: {doc.get('source', 'N/A')}")
    print(f"Text: {doc.get('text', 'N/A')[:300]}...")

if len(retrieved) == 3 and all('text' in d for d in retrieved):
    print("\n✅ Retrieval test passed!")
else:
    print("\n❌ Check your retrieval implementation")

Query: How do I fine-tune a transformer model?

Retrieved 3 documents:

--- Document 1 (Score: 0.7484297752380371) ---
Source: huggingface/blog/blob/main/ray-rag.md
Text: ects/rag/finetune_rag_ray.sh) for faster distributed fine-tuning, you can leverage RAG for retrieval-based generation on your own knowledge-intensive tasks.


Also, hyperparameter tuning is another aspect of transformer fine tuning and can have [huge impacts on accuracy](https://medium.com/distribut...

--- Document 2 (Score: 0.7302744388580322) ---
Source: huggingface/blog/blob/main/lewis-tunstall-interview.md
Text: n try to integrate it into your application. 

So what I've been working on for the last few months on the transformers library is providing the functionality to export these models into a format that lets you run them much more efficiently using tools that we have at Hugging Face, but also just gen...

--- Document 3 (Score: 0.7254992723464966) ---
Source: huggingface/course/blob/main/chapters/en/chapter

## 6.1 Retrieval quality benchmark

Use a small labeled query set to measure Recall@K / MRR-style retrieval quality. The expected terms are intentionally simple and auditable.


In [ ]:
import pandas as pd

# ============================================================
# Stage 5.1 — Retrieval Benchmark
# ============================================================

retrieval_benchmark = [
    {
        "query": "How do I fine-tune a transformer model?",
        "expected_terms": ["fine-tun", "transformers"]
    },
    {
        "query": "How do I load a dataset from Hugging Face?",
        "expected_terms": ["dataset", "load_dataset"]
    },
    {
        "query": "What is Gradio used for?",
        "expected_terms": ["gradio"]
    },
    {
        "query": "How do I create a Hugging Face endpoint?",
        "expected_terms": ["endpoint"]
    },
]


# ============================================================
# Stage 5.2 — Precision@K / Recall@K
# ============================================================

def retrieval_metrics_at_k(query_item, k=5):
    """
    Retrieve the top-k documents and calculate keyword-based
    retrieval metrics.

    A retrieved chunk is considered relevant when it contains
    at least one expected term.

    Returns:
        dict containing precision, recall, hit, first relevant
        rank, and retrieved documents.
    """

    docs = retrieve_documents(
        query_item["query"],
        milvus_client,
        COLLECTION_NAME,
        embedding_model,
        top_k=k
    )

    expected_terms = [
        term.lower()
        for term in query_item["expected_terms"]
    ]

    # --------------------------------------------------------
    # Determine relevance of each retrieved chunk
    # --------------------------------------------------------

    relevant_flags = []

    for doc in docs:
        doc_text = doc["text"].lower()

        is_relevant = any(
            term in doc_text
            for term in expected_terms
        )

        relevant_flags.append(is_relevant)

    # --------------------------------------------------------
    # Precision@K
    # --------------------------------------------------------

    retrieved_count = len(docs)
    relevant_retrieved = sum(relevant_flags)

    if retrieved_count > 0:
        precision = relevant_retrieved / retrieved_count
    else:
        precision = 0.0

    # --------------------------------------------------------
    # Recall@K
    # --------------------------------------------------------

    combined_text = " ".join(
        doc["text"].lower()
        for doc in docs
    )

    found_terms = [
        term
        for term in expected_terms
        if term in combined_text
    ]

    if expected_terms:
        recall = len(found_terms) / len(expected_terms)
    else:
        recall = 0.0

    # --------------------------------------------------------
    # Hit@K
    # --------------------------------------------------------

    hit = relevant_retrieved > 0

    # --------------------------------------------------------
    # First relevant result rank
    # --------------------------------------------------------

    first_relevant_rank = None

    for rank, is_relevant in enumerate(
        relevant_flags,
        start=1
    ):
        if is_relevant:
            first_relevant_rank = rank
            break

    return {
        "precision": precision,
        "recall": recall,
        "hit": hit,
        "first_relevant_rank": first_relevant_rank,
        "docs": docs
    }


# ============================================================
# Run Retrieval Evaluation
# ============================================================

K = 5

evaluation_results = []

for item in retrieval_benchmark:

    result = retrieval_metrics_at_k(
        item,
        k=K
    )

    evaluation_results.append({
        "Query": item["query"],
        "Precision@5": result["precision"],
        "Recall@5": result["recall"],
        "Hit@5": int(result["hit"]),
        "First Relevant Rank": result["first_relevant_rank"]
    })

    print(f"Q: {item['query']}")
    print(f"  Precision@5: {result['precision']:.3f}")
    print(f"  Recall@5:    {result['recall']:.3f}")
    print(f"  Hit@5:       {result['hit']}")
    print(
        f"  First rank:  "
        f"{result['first_relevant_rank']}"
    )
    print("-" * 80)


# ============================================================
# Retrieval Results DataFrame
# ============================================================

retrieval_results_df = pd.DataFrame(
    evaluation_results
)

display(retrieval_results_df)


# ============================================================
# Aggregate Retrieval Metrics
# ============================================================

precision_at_5 = (
    retrieval_results_df["Precision@5"].mean()
)

recall_at_5 = (
    retrieval_results_df["Recall@5"].mean()
)

hit_at_5 = (
    retrieval_results_df["Hit@5"].mean()
)


# ============================================================
# MRR@5
# ============================================================

reciprocal_ranks = []

for rank in retrieval_results_df[
    "First Relevant Rank"
]:

    if pd.isna(rank):
        reciprocal_ranks.append(0.0)
    else:
        reciprocal_ranks.append(
            1.0 / rank
        )

mrr_at_5 = (
    sum(reciprocal_ranks)
    / len(reciprocal_ranks)
)


print("\n" + "=" * 80)
print("RETRIEVAL EVALUATION")
print("=" * 80)

print(f"Average Precision@5: {precision_at_5:.3f}")
print(f"Average Recall@5:    {recall_at_5:.3f}")
print(f"Hit@5:               {hit_at_5:.3f}")
print(f"MRR@5:               {mrr_at_5:.3f}")


# ============================================================
# Stage 5.4 — Evaluation Targets
# ============================================================

PRECISION_TARGET = 0.60
RECALL_TARGET = 0.80
ANSWER_RELEVANCE_TARGET = 0.70
HALLUCINATION_TARGET = 0.30


# ============================================================
# Get Opik Results
# ============================================================

# Your notebook's evaluate_with_opik() function returns rows
# containing:
#
#   answer_relevance
#   hallucination
#
# Run:
#
#     opik_results = evaluate_with_opik(rag_results)
#
# before this section.

if opik_results:

    answer_relevance_scores = [
        row["answer_relevance"]
        for row in opik_results
    ]

    hallucination_scores = [
        row["hallucination"]
        for row in opik_results
    ]

    avg_answer_relevance = (
        sum(answer_relevance_scores)
        / len(answer_relevance_scores)
    )

    avg_hallucination = (
        sum(hallucination_scores)
        / len(hallucination_scores)
    )

else:

    avg_answer_relevance = None
    avg_hallucination = None

    print(
        "\n⚠️ Opik results are not available."
    )


# ============================================================
# Final Evaluation Summary
# ============================================================

summary_rows = [
    {
        "Metric": "Precision@5",
        "Score": precision_at_5,
        "Target": PRECISION_TARGET,
        "Direction": ">=",
        "Status": (
            "PASS"
            if precision_at_5 >= PRECISION_TARGET
            else "FAIL"
        )
    },
    {
        "Metric": "Recall@5",
        "Score": recall_at_5,
        "Target": RECALL_TARGET,
        "Direction": ">=",
        "Status": (
            "PASS"
            if recall_at_5 >= RECALL_TARGET
            else "FAIL"
        )
    }
]


# ------------------------------------------------------------
# Add Opik metrics only when available
# ------------------------------------------------------------

if avg_answer_relevance is not None:

    summary_rows.append({
        "Metric": "Answer Relevance",
        "Score": avg_answer_relevance,
        "Target": ANSWER_RELEVANCE_TARGET,
        "Direction": ">=",
        "Status": (
            "PASS"
            if avg_answer_relevance >= ANSWER_RELEVANCE_TARGET
            else "FAIL"
        )
    })


if avg_hallucination is not None:

    summary_rows.append({
        "Metric": "Hallucination",
        "Score": avg_hallucination,
        "Target": HALLUCINATION_TARGET,
        "Direction": "<=",
        "Status": (
            "PASS"
            if avg_hallucination <= HALLUCINATION_TARGET
            else "FAIL"
        )
    })


evaluation_summary = pd.DataFrame(
    summary_rows
)


# ============================================================
# Format and Display
# ============================================================

evaluation_summary["Score"] = (
    evaluation_summary["Score"].round(3)
)

evaluation_summary["Target"] = (
    evaluation_summary["Target"].round(3)
)

evaluation_summary = evaluation_summary.set_index(
    "Metric"
)

display(evaluation_summary)


# ============================================================
# Overall Result
# ============================================================

all_passed = (
    evaluation_summary["Status"] == "PASS"
).all()

print("\n" + "=" * 80)
print("OVERALL EVALUATION")
print("=" * 80)

if all_passed:
    print("✅ All evaluation targets passed.")
else:
    print("⚠️ One or more evaluation targets were not met.")

Q: How do I fine-tune a transformer model?
  Precision@5: 1.000
  Recall@5:    1.000
  Hit@5:       True
  First rank:  1
--------------------------------------------------------------------------------
Q: How do I load a dataset from Hugging Face?
  Precision@5: 1.000
  Recall@5:    1.000
  Hit@5:       True
  First rank:  1
--------------------------------------------------------------------------------
Q: What is Gradio used for?
  Precision@5: 1.000
  Recall@5:    1.000
  Hit@5:       True
  First rank:  1
--------------------------------------------------------------------------------
Q: How do I create a Hugging Face endpoint?
  Precision@5: 0.400
  Recall@5:    1.000
  Hit@5:       True
  First rank:  1
--------------------------------------------------------------------------------


,Query,Precision@5,Recall@5,Hit@5,First Relevant Rank
0,How do I fine-tune a transformer model?,1.0,1.0,1,1
1,How do I load a dataset from Hugging Face?,1.0,1.0,1,1
2,What is Gradio used for?,1.0,1.0,1,1
3,How do I create a Hugging Face endpoint?,0.4,1.0,1,1



RETRIEVAL EVALUATION
Average Precision@5: 0.850
Average Recall@5:    1.000
Hit@5:               1.000
MRR@5:               1.000


,Score,Target,Direction,Status
Metric,,,,
Precision@5,0.850,0.6,>=,PASS
Recall@5,1.000,0.8,>=,PASS
Answer Relevance,0.725,0.7,>=,PASS
Hallucination,0.000,0.3,<=,PASS



OVERALL EVALUATION
✅ All evaluation targets passed.


## 7. Generation

Generate answers using Microsoft Phi-3-mini-4k-instruct/Qwen.

### Your Task
Implement the `generate_answer` function that:
1. Combines retrieved documents into a context string
2. Formats the prompt using the provided template
3. Generates an answer using the language model
4. Returns a structured result dictionary

### Hints
- Join document texts with newlines to create context
- Use the PROMPT_TEMPLATE.format() to fill in context and question
- Call the generator pipeline with appropriate parameters

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, pipeline
import torch

LLM_MODEL = "microsoft/Phi-3-mini-4k-instruct"

print(f"Loading tokenizer: {LLM_MODEL}")
tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL,
    trust_remote_code=True
)

print("Loading model...")
model_kwargs = {"device_map": "auto","trust_remote_code": True,}

# Prefer float16 on CUDA; otherwise let Transformers choose a CPU-safe dtype.
if torch.cuda.is_available():
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(LLM_MODEL,**model_kwargs)

# The model config already uses rope_scaling=None for the 4k model.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("✅ Tokenizer and model loaded successfully.")
print("Model device:", getattr(model, "device", "managed by device_map"))


Loading tokenizer: microsoft/Phi-3-mini-4k-instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

✅ Tokenizer and model loaded successfully.
Model device: cuda:0


In [ ]:
# Configure generation once and pass it to the pipeline.
generation_config = GenerationConfig(
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    generation_config=generation_config
)

print("✅ Generator created successfully!")


Device set to use cuda:0


✅ Generator created successfully!


In [ ]:
PROMPT_TEMPLATE = """You are a helpful assistant answering questions using the provided context.
Use only the information given in the <context> to answer the <question>.
Do not make up or assume information that is not present in the context.
If the context does not contain enough information to answer the question, say:
"I don't have enough information to answer this question."

<context>
{context}
</context>

<question>
{question}
</question>

Answer:"""

In [ ]:
import re

def clean_answer(text: str) -> str:
    """Truncate generation at the first sign of the model drifting into a new turn/instruction."""
    cut_markers = [
        r"\n#+\s", r"\n###", r"\nQ:", r"\nQuestion:", r"\n<context>",
        r"\nYour task", r"\nInstruction:", r"\n\d+\.\s.*\?",  # numbered follow-up questions
    ]
    earliest = len(text)
    for pattern in cut_markers:
        m = re.search(pattern, text)
        if m:
            earliest = min(earliest, m.start())
    return text[:earliest].strip()

In [ ]:
STOPWORDS = {
    "the","a","an","is","are","was","were","be","been","being","to","of","in",
    "on","at","for","with","and","or","but","what","how","do","does","did",
    "i","you","it","this","that","tomorrow","will",
}

In [ ]:
def semantic_guardrail(
    query: str,
    retrieved_docs: List[Dict],
    min_score: float = 0.35,
    min_overlap: float = 0.15,
) -> bool:
    """

    """
    if not retrieved_docs:
        return True

    best_score = max(doc.get("score", 0.0) for doc in retrieved_docs)
    if best_score < min_score:
        return True

    query_terms = {w for w in re.findall(r"\w+", query.lower()) if w not in STOPWORDS}
    context_text = " ".join(d["text"] for d in retrieved_docs).lower()
    context_terms = {w for w in re.findall(r"\w+", context_text) if w not in STOPWORDS}

    if not query_terms:
        return True

    matched = query_terms & context_terms
    overlap = len(query_terms & context_terms) / len(query_terms)
    if len(query_terms) <= 3:
        return matched != query_terms
    return overlap < min_overlap

In [ ]:
# ============================================================
# TODO: IMPLEMENT GENERATION
# ============================================================
def generate_answer(
    query: str,
    retrieved_docs: List[Dict],
    generator,
    max_new_tokens: int = 256,
    min_retrieval_score: float = 0.25,
    min_overlap: float = 0.15,
    deterministic: bool = True,
) -> Dict:
    """Generate a grounded answer with an explicit insufficient-context guardrail."""
    if not retrieved_docs:
        answer = "I don't have enough information to answer this question."
        return {"query": query, "answer": answer, "context": "", "retrieved_docs": [] ,"guardrail_triggered": True, }

    best_score = max(doc.get("score", 0.0) for doc in retrieved_docs)
    context = "\n\n".join(doc["text"] for doc in retrieved_docs)

    if semantic_guardrail(query, retrieved_docs, min_score=min_retrieval_score,    min_overlap=min_overlap):
        answer = "I don't have enough information to answer this question."
        return {
            "query": query,
            "answer": answer,
            "context": context,
            "retrieved_docs": retrieved_docs,
            "guardrail_triggered": True,
        }

    prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    gen_kwargs = dict(max_new_tokens=max_new_tokens, return_full_text=False)
    if deterministic:
        gen_kwargs["do_sample"] = False          # reproducible for grading/eval
        gen_kwargs["temperature"] = None
        gen_kwargs["top_p"] = None
    else:
        gen_kwargs.update(do_sample=True, temperature=0.7, top_p=0.9)


    outputs = generator(prompt, **gen_kwargs )
    raw_answer = outputs[0]["generated_text"].strip()
    answer = clean_answer(raw_answer)

    # Retry once if the output is degenerate (e.g. just a leaked prompt tag)
    if len(answer) < 20:
      outputs = generator(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.3, top_p=0.9, return_full_text=False)
      answer = clean_answer(outputs[0]["generated_text"].strip())


    return {
        "query": query,
        "answer": answer,
        "context": context,
        "retrieved_docs": retrieved_docs,
        "guardrail_triggered": False,
    }


In [ ]:
# Test generation
test_query = "How do I fine-tune a transformer model?"

# Retrieve relevant documents
retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

# Generate answer
result = generate_answer(
    query=test_query,
    retrieved_docs=retrieved,
    generator=generator,
    max_new_tokens=256,
    min_retrieval_score=0.35,
    min_overlap=0.15,
    deterministic=True,   #
)

print(f"Question: {result['query']}")
print(f"\nAnswer: {result['answer']}")
print(f"\nGuardrail triggered: {result['guardrail_triggered']}")

if result['answer'] and len(result['answer']) > 10:
    print("\n✅ Generation test passed!")
else:
    print("\n❌ Check your generation implementation")

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


Question: How do I fine-tune a transformer model?

Answer: To fine-tune a transformer model, you can follow these steps:

1. Choose a pretrained model: Select a pretrained model from the Hugging Face Hub, such as BERT, GPT-2, or T5.

2. Prepare your data: Organize your data into a suitable format for training, such as a CSV file or a JSON file.

3. Install the required libraries: Install the transformers library and any other necessary libraries, such as PyTorch or TensorFlow.

4. Load the pretrained model and tokenizer: Use the transformers library to load the pretrained model and tokenizer.

5. Fine-tune the model: Use the fine-tuning functionality provided by the transformers library to train the model on your data. This typically involves creating a training loop, defining a loss function, and updating the model's weights using an optimizer.

6. Evaluate the model: After fine-tuning, evaluate the model's performance on a validation set or test set to ensure that it has learned the 

In [ ]:
# Complete RAG pipeline function
def rag_query(query, client, collection_name, embedding_model, generator,
              top_k=5, max_new_tokens=256, min_retrieval_score=0.35,
              min_overlap=0.15, deterministic=True) -> Dict:
    retrieved_docs = retrieve_documents(query, client, collection_name, embedding_model, top_k=top_k)

    return generate_answer(query, retrieved_docs, generator, max_new_tokens=max_new_tokens,
                            min_retrieval_score=min_retrieval_score, min_overlap=min_overlap,
                            deterministic=deterministic)

In [ ]:
# Test complete pipeline with multiple in-domain and one out-of-scope query
end_to_end_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?",
    "What is the weather on Mars tomorrow?",  # deliberate out-of-domain guardrail test
]

end_to_end_results = []
for query in end_to_end_queries:
    print()
    print("=" * 70)

    result = rag_query(query=query, client=milvus_client, collection_name=COLLECTION_NAME,
                        embedding_model=embedding_model, generator=generator, top_k=5)

    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")
    print(f"Guardrail triggered: {result['guardrail_triggered']}")

    end_to_end_results.append(result)


Q: What is the Trainer class in transformers?
A: The Trainer class in transformers is a high-level API that simplifies the training process of transformer models. It abstracts away the complexities of setting up the training loop, managing the GPUs, and handling the training metrics. The Trainer class provides a convenient way to train models using the Hugging Face Transformers library.
Guardrail triggered: False



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q: How do I load a dataset from HuggingFace?
A: <context>
Guardrail triggered: False

Q: What is Gradio used for?
A: Gradio is an open-source Python library used for creating interactive machine learning (ML) models. It allows users to build and share user-friendly interfaces for their ML models, making it easier for non-technical users to interact with and understand the models. Gradio provides a simple and intuitive way to create web-based interfaces for ML models, allowing users to input data, receive predictions, and visualize results in real-time.

Gradio supports a wide range of ML models, including image classification, text classification, and natural language processing models. It also provides various components and widgets that can be used to create custom interfaces, such as sliders, dropdowns, and text inputs.

By using Gradio, developers can easily create interactive interfaces for their ML models, making it easier for users to explore and understand the models' capabilit

## Dependency note

This notebook intentionally uses one pinned Hugging Face stack: `transformers==4.48.3`, `tokenizers==0.21.0`, `accelerate==1.4.0`, and `sentence-transformers==3.4.1`. Restart the Colab runtime after the installation cell and do not uninstall/reinstall these packages later in the same kernel.


## 8. Automated QA with Opik

Evaluate AnswerRelevance and Hallucination using Opik LLM-as-a-judge metrics. A judge credential (for example `OPENAI_API_KEY`) may be required by the configured provider.


In [ ]:
from statistics import mean
import os

def evaluate_with_opik(results):
    """Run AnswerRelevance and Hallucination using an explicit judge model."""
    if not os.environ.get("OPIK_API_KEY"):
        print("ℹ️ OPIK_API_KEY not configured; skipping Opik evaluation.")
        return None
    if not os.environ.get("OPENAI_API_KEY"):
        print("❌ OPENAI_API_KEY not configured; skipping Opik evaluation.")
        return None
    try:
        from opik.evaluation.metrics import (
            AnswerRelevance,
            Hallucination
        )
    except Exception as exc:
        print("❌ Could not import Opik evaluation metrics:", exc)
        return None

    # ---------------------------------------------------------
    # Explicit judge model
    # ---------------------------------------------------------
    JUDGE_MODEL = "gpt-4o-mini"

    print(f"✅ Opik judge model: {JUDGE_MODEL}")

    relevance_metric = AnswerRelevance(model=JUDGE_MODEL)
    hallucination_metric = Hallucination(model=JUDGE_MODEL)

    rows = []

    for result in results:

        context_list = [doc["text"] for doc in result["retrieved_docs"]]

        try:

            relevance = relevance_metric.score(input=result["query"],output=result["answer"],context=context_list,)

            hallucination = hallucination_metric.score(input=result["query"],output=result["answer"],  context=context_list,
            )

            row = {
                "query": result["query"],
                "answer_relevance": relevance.value,
                "hallucination": hallucination.value,
                "relevance_reason": relevance.reason,
                "hallucination_reason": hallucination.reason,
            }
            rows.append(row)
            print()
            print("=" * 80)
            print("Q:", row["query"])
            print(f"Answer relevance: {row['answer_relevance']:.3f}")
            print(f"Hallucination:    {row['hallucination']:.3f}")
            print("Relevance reason:",row["relevance_reason"])
            print("Hallucination reason:",row["hallucination_reason"])

        except Exception as exc:
            print()
            print("⚠️ Opik scoring failed")
            print("Query:", result["query"])
            print("Error:", exc)

    # ---------------------------------------------------------
    # Aggregate results
    # ---------------------------------------------------------
    if not rows:
        print()
        print("❌ No Opik scores were generated.")
        return None

    avg_relevance = mean(
        row["answer_relevance"]
        for row in rows
    )

    avg_hallucination = mean(
        row["hallucination"]
        for row in rows
    )

    print()
    print("=" * 80)
    print("AGGREGATE QA RESULTS")
    print("=" * 80)

    print(f"Average AnswerRelevance: {avg_relevance:.3f}")

    print(f"Average Hallucination:   {avg_hallucination:.3f}")

    print()
    print("Interpretation:")
    print("  AnswerRelevance → higher is better.")
    print("  Hallucination   → lower is better.")
    return rows

In [ ]:
def precision_recall_at_k(query_item, retrieve_fn, k=5):
    """Precision@k: fraction of top-k results that are relevant.
    Recall@k: whether any relevant chunk was found in top-k (single-relevant-doc case)."""
    retrieved = retrieve_fn(query_item["query"], top_k=k)
    relevant_flags = [
        any(term.lower() in doc["text"].lower() for term in query_item["expected_terms"])
        for doc in retrieved
    ]
    precision = sum(relevant_flags) / len(relevant_flags) if relevant_flags else 0.0
    recall = 1.0 if any(relevant_flags) else 0.0
    return precision, recall

precisions, recalls = [], []
for item in retrieval_benchmark:
    p, r = precision_recall_at_k(item, lambda q, top_k: retrieve_documents(
        query=q, client=milvus_client, collection_name=COLLECTION_NAME,
        embedding_model=embedding_model, top_k=top_k), k=5)
    precisions.append(p); recalls.append(r)
    print(f"Q: {item['query']} | Precision@5={p:.2f} | Recall@5={r:.2f}")

print(f"\nMean Precision@5: {sum(precisions)/len(precisions):.3f}")
print(f"Mean Recall@5:    {sum(recalls)/len(recalls):.3f}")

Q: How do I fine-tune a transformer model? | Precision@5=1.00 | Recall@5=1.00
Q: How do I load a dataset from Hugging Face? | Precision@5=1.00 | Recall@5=1.00
Q: What is Gradio used for? | Precision@5=1.00 | Recall@5=1.00
Q: How do I create a Hugging Face endpoint? | Precision@5=0.40 | Recall@5=1.00

Mean Precision@5: 0.850
Mean Recall@5:    1.000


In [ ]:
assert len(embeddings) == len(chunks)
assert all(len(e) == EMBEDDING_DIM for e in embeddings)

print(f"Chunks: {len(chunks)}")
print(f"Embeddings: {len(embeddings)}")
print(f"Dimension: {EMBEDDING_DIM}")

Chunks: 5378
Embeddings: 5378
Dimension: 384


In [ ]:
def diagnose_failures(rows, hallucination_threshold=0.5):
    print("=" * 80)
    print("FAILURE DIAGNOSIS")
    print("=" * 80)
    for row in rows:
        if row["hallucination"] >= hallucination_threshold or row["answer_relevance"] < 0.5:
            reason = row["hallucination_reason"] or row["relevance_reason"]
            # reason may be a real list or a stringified list — normalize it
            if isinstance(reason, list):
                reason_text = reason[0] if reason else "No reason provided."
            else:
                reason_text = str(reason).strip("[]'\" ")
            print(f"\n⚠️ Q: {row['query']}")
            print(f"   Relevance={row['answer_relevance']:.2f}, Hallucination={row['hallucination']:.2f}")
            print(f"   Likely cause: {reason_text}")

In [ ]:
def retrieve_fn(query, top_k):
    return retrieve_documents(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=top_k,
    )

precisions, recalls = [], []
for item in retrieval_benchmark:
    p, r = precision_recall_at_k(item, retrieve_fn, k=5)
    precisions.append(p)
    recalls.append(r)
    print(f"Q: {item['query']} | Precision@5={p:.2f} | Recall@5={r:.2f}")

print(f"\nMean Precision@5: {sum(precisions)/len(precisions):.3f}")
print(f"Mean Recall@5:    {sum(recalls)/len(recalls):.3f}")

Q: How do I fine-tune a transformer model? | Precision@5=1.00 | Recall@5=1.00
Q: How do I load a dataset from Hugging Face? | Precision@5=1.00 | Recall@5=1.00
Q: What is Gradio used for? | Precision@5=1.00 | Recall@5=1.00
Q: How do I create a Hugging Face endpoint? | Precision@5=0.40 | Recall@5=1.00

Mean Precision@5: 0.850
Mean Recall@5:    1.000


In [ ]:
opik_results = evaluate_with_opik(end_to_end_results)

if opik_results:
    diagnose_failures(opik_results)

✅ Opik judge model: gpt-4o-mini

Q: What is the Trainer class in transformers?
Answer relevance: 1.000
Hallucination:    0.000
Relevance reason: The answer accurately and concisely describes the Trainer class in transformers, addressing the user’s question directly. It highlights its purpose and functionalities, which aligns perfectly with the context provided that discusses the use of the `Trainer` class for model training in the Hugging Face Transformers library.
Hallucination reason: ['The OUTPUT accurately describes the role and purpose of the Trainer class in the context of the Transformers library, which was mentioned in the CONTEXT. It does not introduce new information or contradict what is provided.']

Q: How do I load a dataset from HuggingFace?
Answer relevance: 0.900
Hallucination:    0.000
Relevance reason: The answer provides a detailed explanation of how to load a dataset from HuggingFace, including specific code examples and instructions, which directly addresses the us

- `microsoft/Phi-3-mini-4k-instruct` (3.8B params) was chosen over `Qwen2-1.5B-Instruct` for stronger instruction-following on grounded QA, at the cost of higher memory/latency than the smaller Qwen alternative — acceptable for this notebook's batch/demo use case but worth revisiting for low-latency production serving.


## 9. Rubric Coverage Checklist

| Evaluation criterion | Evidence in notebook |
|---|---|
| Data processing & chunking | Sliding window, validated overlap, character offsets, source/doc/chunk lineage metadata |
| Vectorization & memory management | Batched SentenceTransformer encoding + normalized embeddings |
| Vector DB architecture & persistence | Explicit Milvus schema, AUTOINDEX, `IP` metric, Milvus Lite local DB, insert batching, persistence verification |
| Retrieval precision | Normalized-query IP search, `top_k`, scores, Recall@5 and MRR@5 benchmark |
| Grounded synthesis & hallucination control | Context-only prompt, insufficient-context guardrail, out-of-scope test |
| Pipeline integration & reasoning | End-to-end retrieve → generate flow with multiple technical queries |
| Automated QA & metric interpretation | Opik AnswerRelevance + Hallucination scores, reasons, and aggregate interpretation |
| Technical rigor & reproducibility | Pinned dependencies, runtime restart note, environment verification, documented design choices |

### Design trade-offs
- `chunk_size=1000` and `overlap=200` provide local semantic continuity while keeping retrieval units manageable.
- `BAAI/bge-small-en-v1.5` provides 384-dimensional vectors with low memory/latency cost suitable for a notebook demonstration.
- Normalizing embeddings and using Milvus Inner Product makes the search score equivalent to cosine similarity.
- Batched inserts/embeddings reduce peak memory usage.
- The guardrail prevents generation when the retrieved evidence is weak.
